# Load and Prepare Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
from google.colab import files
uploaded = files.upload()

Saving ghana-ghana-premier-league-teams-2025-to-2026-stats.csv to ghana-ghana-premier-league-teams-2025-to-2026-stats.csv


In [10]:
df = pd.read_csv("ghana-ghana-premier-league-teams-2025-to-2026-stats.csv")


In [13]:
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Columns: 293 entries, team_name to over145_corners_percentage
dtypes: float64(36), int64(253), object(4)
memory usage: 41.3+ KB


,team_name,common_name,season,country,matches_played,matches_played_home,matches_played_away,suspended_matches,wins,wins_home,...,goals_conceded_min_61_to_70,goals_conceded_min_71_to_80,goals_conceded_min_81_to_90,draw_percentage_overall,draw_percentage_home,draw_percentage_away,loss_percentage_ovearll,loss_percentage_home,loss_percentage_away,over145_corners_percentage
0,Asante Kotoko FC,Asante Kotoko,2025/2026,Ghana,31,15,16,0,11,8,...,1,2,1,32,33,31,32,13,50,7
1,Bechem United FC,Bechem United,2025/2026,Ghana,33,17,16,0,12,10,...,3,0,8,24,29,19,39,12,69,5
2,Hearts of Oak SC,Hearts of Oak,2025/2026,Ghana,33,16,17,0,13,6,...,3,1,3,45,63,29,15,0,29,10
3,Berekum Chelsea FC,Berekum Chelsea,2025/2026,Ghana,34,17,17,0,14,12,...,4,4,7,24,18,29,35,12,59,7
4,Medeama SC,Medeama,2025/2026,Ghana,34,17,17,0,17,12,...,3,3,3,32,24,41,18,6,29,7


# Check missing values

In [14]:
df.isnull().sum()

,0
team_name,0
common_name,0
season,0
country,0
matches_played,0
...,...
draw_percentage_away,0
loss_percentage_ovearll,0
loss_percentage_home,0
loss_percentage_away,0


# Remove duplicates

In [15]:
df.duplicated().sum()

np.int64(0)

# 1. Who are the current title contenders and relegation risks?

# Build a League Performance Index

In [16]:
df["points_proxy"] = (df["wins"] * 3) + (df["draw_percentage_overall"] * 0.1) - (df["loss_percentage_ovearll"] * 0.2)

# Identify Title Contenders

In [17]:
contenders = df.sort_values("points_proxy", ascending=False)

contenders[["team_name", "wins", "draw_percentage_overall", "loss_percentage_ovearll", "points_proxy"]].head(5)

,team_name,wins,draw_percentage_overall,loss_percentage_ovearll,points_proxy
4,Medeama SC,17,32,18,50.6
12,Bibiani Gold Stars FC,18,9,36,47.7
5,Dreams FC,15,21,33,40.5
2,Hearts of Oak SC,13,45,15,40.5
3,Berekum Chelsea FC,14,24,35,37.4


# Identify relegation risk teams

In [18]:
relegation = df.sort_values("points_proxy", ascending=True)

relegation[["team_name", "wins", "draw_percentage_overall", "loss_percentage_ovearll", "points_proxy"]].head(5)

,team_name,wins,draw_percentage_overall,loss_percentage_ovearll,points_proxy
8,Techiman Eleven Wonders FC,2,12,82,-9.2
15,Hohoe United FC,7,30,47,14.6
11,Young Apostles FC,11,27,39,27.9
0,Asante Kotoko FC,11,32,32,29.8
17,Nations Football Club,12,24,41,30.2


In [19]:
import plotly.express as px

fig = px.bar(
    df.sort_values("points_proxy", ascending=False),
    x="team_name",
    y="points_proxy",
    title="Ghana Premier League - Title Race Index",
    text="points_proxy"
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Question 2: Do teams rely heavily on home advantage?

# Create Home Advantage Index

In [20]:
df["home_advantage"] = df["wins_home"] - df["wins_away"]
df["home_dependency_ratio"] = df["wins_home"] / (df["wins_home"] + df["wins_away"] + 1)

# Strong Home Advantage Teams

In [21]:
df.sort_values("home_advantage", ascending=False)[
    ["team_name", "wins_home", "wins_away", "home_advantage"]
].head(5)

,team_name,wins_home,wins_away,home_advantage
14,Basake Holy Stars FC,12,1,11
7,Karela FC,12,1,11
10,FC Samartex 1996,11,1,10
12,Bibiani Gold Stars FC,14,4,10
6,Heart of Lions FC,11,1,10


# Weak Home Dependency / Strong Away Teams

In [22]:
df.sort_values("home_advantage", ascending=True)[
    ["team_name", "wins_home", "wins_away", "home_advantage"]
].head(5)

,team_name,wins_home,wins_away,home_advantage
2,Hearts of Oak SC,6,7,-1
8,Techiman Eleven Wonders FC,2,0,2
15,Hohoe United FC,5,2,3
9,Gamba All Blacks,8,4,4
0,Asante Kotoko FC,8,3,5


# Question 3: When do teams become defensively vulnerable?

# Build total late-game vulnerability

In [23]:
late_cols = [
    "goals_conceded_min_61_to_70",
    "goals_conceded_min_71_to_80",
    "goals_conceded_min_81_to_90"
]

df["late_game_conceded"] = df[late_cols].sum(axis=1)

# Identify most vulnerable teams

In [24]:
df.sort_values("late_game_conceded", ascending=False)[
    ["team_name"] + late_cols + ["late_game_conceded"]
].head(5)

,team_name,goals_conceded_min_61_to_70,goals_conceded_min_71_to_80,goals_conceded_min_81_to_90,late_game_conceded
8,Techiman Eleven Wonders FC,11,9,12,32
3,Berekum Chelsea FC,4,4,7,15
13,Vision FC,4,5,6,15
11,Young Apostles FC,4,3,8,15
7,Karela FC,4,7,3,14


# Late-game vulnerability index

In [25]:
df["late_game_vulnerability_index"] = (
    df["goals_conceded_min_61_to_70"] * 1 +
    df["goals_conceded_min_71_to_80"] * 1.5 +
    df["goals_conceded_min_81_to_90"] * 2
)

In [26]:
df.sort_values("late_game_vulnerability_index", ascending=False)[
    ["team_name", "late_game_vulnerability_index"] + late_cols
].head(5)

,team_name,late_game_vulnerability_index,goals_conceded_min_61_to_70,goals_conceded_min_71_to_80,goals_conceded_min_81_to_90
8,Techiman Eleven Wonders FC,48.5,11,9,12
11,Young Apostles FC,24.5,4,3,8
3,Berekum Chelsea FC,24.0,4,4,7
13,Vision FC,23.5,4,5,6
15,Hohoe United FC,23.5,2,5,7


# Question 4: “What actually drives winning in this league?”

# Correlation with wins

In [27]:
import pandas as pd

numeric_df = df.select_dtypes(include=["int64", "float64"])

correlations = numeric_df.corr()["wins"].sort_values(ascending=False)

correlations.head(15)

,wins
wins,1.000000
win_percentage,0.995470
points_proxy,0.981371
points_per_game,0.959567
points_per_game_home,0.932397
goal_difference_home,0.909397
win_percentage_home,0.866808
wins_home,0.862780
goal_difference,0.855930
goals_scored_home,0.714014


# Build a Win Influence Model

In [28]:
df["win_influence_score"] = (
    df["wins_home"] * 0.5 +
    df["wins_away"] * 0.5
) - (
    df["loss_percentage_ovearll"] * 0.3
)

# Question 5: Which teams are overperforming or underperforming expectations?

# Create expected strength baseline

In [29]:
df["strength_index"] = (
    df["win_percentage"] * 0.4 +
    df["goal_difference"] * 0.3 +
    df["points_per_game"] * 0.3
)

# Compare expected vs actual wins

In [30]:
df["performance_gap"] = df["wins"] - df["strength_index"]

# Identify overperformers

In [31]:
df.sort_values("performance_gap", ascending=False)[
    ["team_name", "wins", "strength_index", "performance_gap"]
].head(5)

,team_name,wins,strength_index,performance_gap
8,Techiman Eleven Wonders FC,2,-12.510,14.510
15,Hohoe United FC,7,4.700,2.300
14,Basake Holy Stars FC,13,13.205,-0.205
1,Bechem United FC,12,13.599,-1.599
17,Nations Football Club,12,14.087,-2.087


# Identify underperformers

In [32]:
df.sort_values("performance_gap", ascending=True)[
    ["team_name", "wins", "strength_index", "performance_gap"]
].head(5)

,team_name,wins,strength_index,performance_gap
4,Medeama SC,17,27.746,-10.746
5,Dreams FC,15,23.574,-8.574
2,Hearts of Oak SC,13,19.392,-6.392
0,Asante Kotoko FC,11,17.117,-6.117
12,Bibiani Gold Stars FC,18,23.419,-5.419
